In [ ]:
import os
import time
import re
import json
import traceback
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ✅ Utilities
def safe_filename(variant):
    return re.sub(r'[<>:"/\\|?*]', "_", variant)

def double_encode(variant):
    return quote(quote(variant, safe=""), safe="")

# 🧪 Parser
def parse_cftr_functional_metrics(html_path, variant_name):
    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    result = {
        "variant_name": variant_name,
        "variant_type_text": None,
        "cftr_quantity_values": [],
        "cftr_function_values": [],
        "quantity_avg": None,
        "function_avg": None,
        "inferred_class": None,
        "final_call": None,
        "penetrance_text": None
    }

    h4_tag = soup.find("h4", string=re.compile(rf"Functional Testing for {variant_name}", re.I))
    if h4_tag:
        p_tag = h4_tag.find_next("p")
        if p_tag:
            result["variant_type_text"] = p_tag.get_text(" ", strip=True)

    table = soup.find("table", class_="missense_table")
    if table:
        for row in table.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) == 4 and variant_name in cells[0].text:
                qty = re.search(r"(\d+(?:\.\d+)?)", cells[2].text)
                func = re.search(r"(\d+(?:\.\d+)?)", cells[3].text)
                if qty:
                    result["cftr_quantity_values"].append(float(qty.group(1)))
                if func:
                    result["cftr_function_values"].append(float(func.group(1)))

    if result["cftr_quantity_values"]:
        result["quantity_avg"] = round(sum(result["cftr_quantity_values"]) / len(result["cftr_quantity_values"]), 2)
    if result["cftr_function_values"]:
        result["function_avg"] = round(sum(result["cftr_function_values"]) / len(result["cftr_function_values"]), 2)

    q = result["quantity_avg"]
    f = result["function_avg"]
    tag = result["variant_type_text"]

    if q == 0:
        result["inferred_class"] = "Class I"
    elif q is not None and f is not None:
        if q < 20 and f < 10:
            result["inferred_class"] = "Class II"
        elif q >= 80 and f < 10:
            result["inferred_class"] = "Class III"
        elif f < 20:
            result["inferred_class"] = "Class IV"
        elif f >= 80:
            result["inferred_class"] = "Likely non-CF"
        else:
            result["inferred_class"] = "Uncertain"
    elif tag and "nonsense variant" in tag.lower():
        result["inferred_class"] = "Class I"
    elif tag and "splice variant" in tag.lower():
        result["inferred_class"] = "Class I or V"

    history = soup.find("div", class_="drop_box")
    if history:
        for row in history.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) >= 2 and variant_name in cells[0].text:
                result["final_call"] = cells[1].text.strip()

    pop_tab = soup.find("div", id="population")
    if pop_tab:
        text = pop_tab.get_text(" ", strip=True)
        match = re.search(rf"{variant_name}.*?likely causes CF.*?\.", text, re.I)
        if match:
            result["penetrance_text"] = match.group(0)

    return result

# 🔁 Batch Scraping
df = pd.read_excel("CFTR2_25September2024.xlsx", sheet_name="CFTR2 variants by legacy name", skiprows=9)
variant_names = [v.strip() for v in df["Variant legacy name"].dropna().unique()]

SAVE_FOLDER = "cftr2_variant_pages"
os.makedirs(SAVE_FOLDER, exist_ok=True)
json_file = "cftr2_variant_classes.json"

if os.path.exists(json_file):
    with open(json_file, encoding="utf-8") as f:
        class_data_all = json.load(f)
else:
    class_data_all = {}

processed = set(class_data_all.keys())
failed = []
variant_log = []

# 🚀 Start Chrome
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# ☑️ Accept CFTR2 terms
print("🔐 Visiting CFTR2 welcome page...")
driver.get("https://cftr2.org/welcome")
time.sleep(1)

checkbox_ids = ["edit-education", "edit-individual", "edit-discuss", "edit-privacy"]
for box_id in checkbox_ids:
    try:
        checkbox = wait.until(EC.presence_of_element_located((By.ID, box_id)))
        driver.execute_script("arguments[0].scrollIntoView(true);", checkbox)
        if not checkbox.is_selected():
            checkbox.click()
        print(f"✅ Checked: {box_id}")
    except:
        print(f"⚠️ Could not find box: {box_id}")

submit_btn = wait.until(EC.element_to_be_clickable((By.ID, "edit-submit-terms")))
submit_btn.click()
print("✅ Submitted agreement.")

# 🔄 Process all variants
for variant in variant_names:
    variant_clean = variant.strip()

#### NOTE: This section is commented out to avoid reprocessing already handled variants.

# if variant_clean in processed:
#     print(f"⏭️ Skipping: {variant_clean}")
#     continue

    try:
        print(f"\n🔬 Processing: {variant_clean}")
        encoded = double_encode(variant_clean)
        url = f"https://cftr2.org/mutation/scientific/{encoded}"
        print(f"🔗 URL: {url}")
        driver.get(url)
        time.sleep(2)

        html_path = os.path.join(SAVE_FOLDER, f"{safe_filename(variant_clean)}.html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"📄 Saved HTML to: {html_path}")

        info = parse_cftr_functional_metrics(html_path, variant_clean)
        class_data_all[variant_clean] = info
        variant_log.append([variant_clean, url, "Success"])
        print(f"🏷️ Inferred Class: {info['inferred_class']}")

        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(class_data_all, f, indent=2)

    except Exception as e:
        print(f"❌ Error with {variant_clean}: {e}")
        failed.append(variant_clean)
        variant_log.append([variant_clean, url, f"Error: {str(e)}"])
        traceback.print_exc()

# 🛑 Shutdown
driver.quit()
with open("cftr2_variant_failed_log.txt", "w") as f:
    f.write("\n".join(failed))

with open("cftr2_variant_url_log.csv", "w", encoding="utf-8") as f:
    import csv
    writer = csv.writer(f)
    writer.writerow(["Variant", "URL", "Status"])
    writer.writerows(variant_log)

print("\n✅ Finished scraping all variants.")
print("🧪 Functional classification saved to cftr2_variant_classes.json")


In [ ]:
import os
import time
import re
import csv
import json
import pandas as pd
from urllib.parse import quote
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Step 1: Load curated class mapping (CSV from cftr-gene.fr or other) ---
def load_curated_classes(csv_path):
    curated = {}
    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name","").strip()
            cftr_class = row.get("Class","").strip()
            if name and cftr_class:
                curated[name.upper()] = cftr_class
    return curated

# --- Step 2: Scrape cftr2.org for functional info if not in curated ---
def safe_filename(name):
    return re.sub(r'[<>:"/\\|?*]', "_", name)

def double_encode(text):
    return quote(quote(text, safe=""), safe="")

# Mechanism-to-class mapping
FUNC_CLASS_MAP = [
    (re.compile(r"no protein|nonsense|frameshift|no cftr protein", re.I), "Class I"),
    (re.compile(r"traffick|misfold|f508del|processing defect", re.I), "Class II"),
    (re.compile(r"gating", re.I), "Class III"),
    (re.compile(r"conductance|reduced conductance", re.I), "Class IV"),
    (re.compile(r"reduced protein|partial splicing|reduced amount", re.I), "Class V"),
]

def infer_class_from_functional_section(soup):
    text = soup.get_text(" ", strip=True)
    for regex, cls in FUNC_CLASS_MAP:
        if regex.search(text):
            return cls
    # Table parsing (optional: more robust, for numeric cutoffs)
    table = None
    for t in soup.find_all("table"):
        hdrs = [th.get_text(strip=True).lower() for th in t.find_all("th")]
        if "cftr quantity" in "".join(hdrs) and "cftr function" in "".join(hdrs):
            table = t
            break
    if table:
        values = []
        for row in table.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) >= 4:
                try:
                    qty = float(re.search(r"(\d+(?:\.\d+)?)", cells[2].text).group(1))
                    func = float(re.search(r"(\d+(?:\.\d+)?)", cells[3].text).group(1))
                    values.append((qty, func))
                except Exception:
                    continue
        # Use rules of thumb (should be customized per literature)
        for qty, func in values:
            if qty == 0:
                return "Class I"
            elif qty < 20 and func < 10:
                return "Class II"
            elif qty >= 80 and func < 10:
                return "Class III"
            elif func < 20:
                return "Class IV"
            elif func >= 80:
                return "Likely non-CF"
    return None

def scrape_and_assign_class(variant, curated_map, driver, save_folder):
    # Try curated map first
    variant_key = variant.upper()
    if variant_key in curated_map:
        return curated_map[variant_key]
    # Otherwise, scrape cftr2
    encoded = double_encode(variant)
    url = f"https://cftr2.org/mutation/scientific/{encoded}"
    driver.get(url)
    time.sleep(2)
    html_path = os.path.join(save_folder, f"{safe_filename(variant)}.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(driver.page_source)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    # Try summary table
    for table in soup.find_all("table"):
        headers = [th.get_text(strip=True).lower() for th in table.find_all("th")]
        if "variant final determination" in headers:
            for row in table.find_all("tr"):
                cells = row.find_all("td")
                if len(cells) >= 2 and variant.upper() in cells[0].get_text(strip=True).upper():
                    call = cells[1].get_text(strip=True)
                    # Map to class if explicit
                    if "cf-causing" in call.lower():
                        # Try to infer from functional section
                        cls = infer_class_from_functional_section(soup)
                        return cls or "CF-causing"
                    elif "non cf-causing" in call.lower():
                        return "Non CF-causing"
                    elif "varying" in call.lower():
                        return "Varying clinical consequence"
    # Try functional section
    cls = infer_class_from_functional_section(soup)
    return cls or "Unknown"

# --- Step 3: Main workflow ---
def main():
    # Load curated class mapping first!
    curated_map = load_curated_classes("cftr_gene_fr_class_table.csv")  # Downloaded from cftr-gene.fr
    
    df = pd.read_excel("CFTR2_25September2024.xlsx", sheet_name="CFTR2 variants by legacy name", skiprows=9)
    variant_names = [v.strip() for v in df["Variant legacy name"].dropna().unique()]
    SAVE_FOLDER = "cftr2_variant_pages"
    os.makedirs(SAVE_FOLDER, exist_ok=True)
    results_json = "cftr2_variant_classes_with_functional_class.json"
    class_data_all = {}

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    driver = webdriver.Chrome(options=options)
    try:
        # Accept terms
        driver.get("https://cftr2.org/welcome")
        time.sleep(2)
        for box_id in ["edit-education", "edit-individual", "edit-discuss", "edit-privacy"]:
            try:
                box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, box_id)))
                if not box.is_selected():
                    box.click()
            except Exception:
                pass
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "edit-submit-terms"))).click()
        time.sleep(1)

        for variant in variant_names:
            try:
                print(f"Processing: {variant}")
                assigned_class = scrape_and_assign_class(variant, curated_map, driver, SAVE_FOLDER)
                class_data_all[variant] = assigned_class
                print(f"Class: {assigned_class}")
            except Exception as e:
                class_data_all[variant] = "Error"
                print(f"Error: {variant} {e}")

    finally:
        driver.quit()

    with open(results_json, "w", encoding="utf-8") as f:
        json.dump(class_data_all, f, indent=2)
    print(f"Saved to {results_json}")

if __name__ == '__main__':
    main()

In [ ]:
import pandas as pd
import json

# Load your spreadsheet
df = pd.read_excel("CFTR2_Enhanced_Classification.csv.xlsx")  # Update with your actual filename

# Optional: rename columns if needed
df.columns = ["Variant", "Description", "Quantity", "Function", "Old_Class", "Reassigned_Class", "Final_Call", "Penetrance"]

# Drop rows with missing variant names
df = df.dropna(subset=["Variant"])

# Build structured dictionary for JSON export
variant_json = []
for _, row in df.iterrows():
    entry = {
        "Variant": str(row["Variant"]).strip(),
        "Description": str(row["Description"]).strip(),
        "Quantity": row.get("Quantity", None),
        "Function": row.get("Function", None),
        "Old_Class": str(row.get("Old_Class", "")).strip(),
        "Reassigned_Class": str(row.get("Reassigned_Class", "")).strip(),
        "Final_Call": str(row.get("Final_Call", "")).strip(),
        "Penetrance": str(row.get("Penetrance", "")).strip(),
    }
    variant_json.append(entry)

# Save to JSON file
with open("cftr_variant_classes.json", "w") as f:
    json.dump(variant_json, f, indent=4)

print("✅ Excel data successfully exported to cftr_variant_classes.json")
